### Get OpenAlex institutions from OpenAlex API
 - Source: https://api.openalex.org/institutions
 - Save to `OpenAlex_institutions.csv`

In [ ]:
import pandas as pd
import requests
import yaml
from tqdm import tqdm

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
base_url = "https://api.openalex.org/institutions?per-page=200&cursor={}"
cursor = "*"
records = []
row_idx = 0

In [ ]:
with tqdm(desc="Processing pages") as pbar:
    while cursor:
        url = base_url.format(cursor)
        response = requests.get(url)
        data = response.json()

        # Stop if an error is encountered
        if 'error' in data:
            pbar.write(f"Error encountered: {data['error']}")
            break

        # Process each result to extract 'id' and 'country_code'
        for item in data.get('results', []):
            records.append({
                'id': item.get('id').split('/')[-1],
                'country_code': item.get('country_code')
            })

        row_idx += 1
        # Update the progress bar and display additional info
        pbar.update(1)
        pbar.set_postfix({'row': row_idx, 'records': len(records)})

        # Update the cursor using the value from 'meta'
        cursor = data.get('meta', {}).get('next_cursor')
        if not cursor:
            break


In [ ]:
institutions_oa = pd.DataFrame(records)
institutions_oa

In [ ]:
institutions_oa = institutions_oa.dropna().drop_duplicates()
institutions_oa  # 102476 rows

In [ ]:
institutions_oa.to_csv(dataset_config['path_other'] + 'OpenAlex_institutions.csv', index=False)